# Walker2d PPO：单 Notebook 完整学习项目

这本 Notebook 包含项目的**全部可执行代码**：环境探索、Actor/Critic、Rollout Buffer、GAE、PPO-Clip、训练、日志、checkpoint、评估、GIF 和断言测试。不依赖仓库中的任何 `.py` 源文件，方便逐单元修改和调试。

> `smoke` 只验证 rollout → GAE → PPO update → checkpoint → evaluation 的接线，不代表策略已经学会行走。完整训练效果必须用真实日志和视频判断。

## 00｜依赖、随机种子和设备

首版默认 CPU。MuJoCo 仿真仍在 CPU，小型 MLP 放到 MPS 不一定更快。若缺包，请先在终端运行 `pip install -r requirements.txt`。

In [ ]:
from __future__ import annotations

import csv
import json
import platform
import random
import sys
import time
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path
from typing import Any

import gymnasium as gym
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import torch
from gymnasium import spaces
from torch import nn
from torch.distributions import Normal

SEED = 1
DEVICE = torch.device("cpu")

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything(SEED)
print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gymnasium": gym.__version__,
    "torch": torch.__version__,
    "device": str(DEVICE),
})

## 01｜环境、17 维观测和 6 维动作

默认观测由 8 个位置量（绝对 x 被删除）和 9 个速度量组成。环境动作是六个铰链上的力矩，不是“目标角度 + PD”。

In [ ]:
OBSERVATION_NAMES = [
    "torso height z", "torso angle",
    "right thigh angle", "right leg angle", "right foot angle",
    "left thigh angle", "left leg angle", "left foot angle",
    "torso x velocity", "torso z velocity", "torso angular velocity",
    "right thigh angular velocity", "right leg angular velocity", "right foot angular velocity",
    "left thigh angular velocity", "left leg angular velocity", "left foot angular velocity",
]

ACTION_NAMES = [
    "right thigh torque", "right leg torque", "right foot torque",
    "left thigh torque", "left leg torque", "left foot torque",
]

env = gym.make("Walker2d-v5")
obs, reset_info = env.reset(seed=SEED)
action = env.action_space.sample()
next_obs, reward, terminated, truncated, info = env.step(action)

print("observation_space:", env.observation_space)
print("action_space:", env.action_space)
print("reset observation:", obs.shape, obs.dtype)
print("network input dtype:", np.asarray(obs, dtype=np.float32).dtype)
print("step tuple:", next_obs.shape, reward, terminated, truncated)
print("reward terms:", {k: info[k] for k in ("reward_forward", "reward_ctrl", "reward_survive")})
print("dt:", env.unwrapped.dt, "=", env.unwrapped.model.opt.timestep, "*", env.unwrapped.frame_skip)

assert obs.shape == (17,) and obs.dtype == np.float64
assert action.shape == (6,) and action.dtype == np.float32
assert np.isclose(reward, info["reward_forward"] + info["reward_ctrl"] + info["reward_survive"])
assert np.isclose(env.unwrapped.dt, env.unwrapped.model.opt.timestep * env.unwrapped.frame_skip)

for i, name in enumerate(OBSERVATION_NAMES):
    print(f"{i:2d}: {name:34s} {obs[i]: .5f}")
env.close()

`info["reward_ctrl"]` 已经是负数，所以总奖励直接写成：

```text
reward = reward_forward + reward_survive + reward_ctrl
```

不要再次减去 `reward_ctrl`。默认一步环境时间为 `4 × 0.002 = 0.008 s`。

## 02｜随机策略、跌倒诊断和 GIF

随机动作通常很快导致躯干高度或角度越界。下面保存短 GIF，并同步画出关键状态。渲染失败不会阻止无界面训练。

In [ ]:
video_path = Path("outputs/random_policy.gif")
video_path.parent.mkdir(parents=True, exist_ok=True)
render_env = gym.make("Walker2d-v5", render_mode="rgb_array")
obs, _ = render_env.reset(seed=SEED)
frames, heights, angles, x_velocities = [], [], [], []
random_return = 0.0

for step in range(200):
    obs, reward, terminated, truncated, info = render_env.step(render_env.action_space.sample())
    frame = render_env.render()
    if frame is not None:
        frames.append(frame)
    heights.append(float(obs[0]))
    angles.append(float(obs[1]))
    x_velocities.append(float(obs[8]))
    random_return += float(reward)
    if terminated or truncated:
        break
render_env.close()

if frames:
    imageio.mimsave(video_path, frames, fps=30, loop=0)
print({
    "return": random_return, "length": len(heights),
    "terminated": terminated, "truncated": truncated,
    "video": str(video_path) if frames else "render unavailable",
})

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(heights, label="torso height")
axes[0].plot(angles, label="torso angle")
axes[0].axhline(0.8, color="gray", linestyle="--", linewidth=1)
axes[0].legend(); axes[0].set_xlabel("environment step")
axes[1].plot(x_velocities, label="x velocity")
axes[1].legend(); axes[1].set_xlabel("environment step")
plt.tight_layout(); plt.show()

## 03｜集中配置和三种运行档位

参数集中在一个 dataclass 中。`smoke` 只做接线验收；`short` 观察趋势；`full` 是较大预算，但仍不预设一定收敛。

In [ ]:
@dataclass(slots=True)
class PPOConfig:
    env_id: str = "Walker2d-v5"
    total_timesteps: int = 200_000
    rollout_steps: int = 2_048
    update_epochs: int = 10
    minibatch_size: int = 64
    gamma: float = 0.99
    gae_lambda: float = 0.95
    clip_coef: float = 0.2
    actor_lr: float = 3e-4
    critic_lr: float = 1e-3
    entropy_coef: float = 0.0
    value_coef: float = 0.5
    max_grad_norm: float = 0.5
    hidden_size: int = 64
    seed: int = 1
    device: str = "cpu"
    eval_episodes: int = 3
    output_dir: Path = Path("runs/baseline")
    forward_reward_weight: float = 1.0
    ctrl_cost_weight: float = 1e-3
    healthy_reward: float = 1.0

    def validate(self) -> None:
        assert self.total_timesteps > 0 and self.rollout_steps > 0
        assert 0 < self.minibatch_size <= self.rollout_steps
        assert 0 <= self.gamma <= 1 and 0 <= self.gae_lambda <= 1
        assert self.clip_coef > 0
        assert self.device in {"cpu", "mps"}

    def env_kwargs(self) -> dict[str, float]:
        return {
            "forward_reward_weight": self.forward_reward_weight,
            "ctrl_cost_weight": self.ctrl_cost_weight,
            "healthy_reward": self.healthy_reward,
        }

    def as_serializable_dict(self) -> dict[str, object]:
        result = asdict(self)
        result["output_dir"] = str(self.output_dir)
        return result


def profile(name: str, **overrides: object) -> PPOConfig:
    profiles = {
        "smoke": PPOConfig(
            total_timesteps=128, rollout_steps=128, update_epochs=1,
            minibatch_size=64, eval_episodes=1, output_dir=Path("runs/smoke"),
        ),
        "short": PPOConfig(
            total_timesteps=20_000, rollout_steps=1_024, update_epochs=5,
            minibatch_size=64, eval_episodes=3, output_dir=Path("runs/short"),
        ),
        "full": PPOConfig(),
    }
    if name not in profiles:
        raise ValueError(f"unknown profile: {name}")
    return replace(profiles[name], **overrides)


print(profile("smoke"))

## 04｜Actor、Critic 和连续动作

Actor 输出因子化高斯分布。六维动作的联合 log probability 必须只沿动作维求和：`[batch, 6] → [batch]`。

In [ ]:
def make_mlp(input_dim: int, output_dim: int, hidden_size: int) -> nn.Sequential:
    return nn.Sequential(
        nn.Linear(input_dim, hidden_size), nn.Tanh(),
        nn.Linear(hidden_size, hidden_size), nn.Tanh(),
        nn.Linear(hidden_size, output_dim),
    )


class GaussianActor(nn.Module):
    # 策略建模 raw action u；环境执行 clip(u, -1, 1)。

    def __init__(self, obs_dim: int, action_dim: int, hidden_size: int = 64) -> None:
        super().__init__()
        self.mean_net = make_mlp(obs_dim, action_dim, hidden_size)
        self.log_std = nn.Parameter(torch.full((action_dim,), -0.5))

    def distribution(self, obs: torch.Tensor) -> Normal:
        mean = self.mean_net(obs)
        return Normal(mean, self.log_std.exp().expand_as(mean))

    @staticmethod
    def joint_log_prob(dist: Normal, raw_action: torch.Tensor) -> torch.Tensor:
        return dist.log_prob(raw_action).sum(dim=-1)

    def sample(self, obs: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        dist = self.distribution(obs)
        raw_action = dist.sample()
        log_prob = self.joint_log_prob(dist, raw_action)
        entropy = dist.entropy().sum(dim=-1)
        return raw_action, log_prob, entropy

    def deterministic(self, obs: torch.Tensor) -> torch.Tensor:
        return self.mean_net(obs)


class ValueCritic(nn.Module):
    def __init__(self, obs_dim: int, hidden_size: int = 64) -> None:
        super().__init__()
        self.value_net = make_mlp(obs_dim, 1, hidden_size)

    def forward(self, obs: torch.Tensor) -> torch.Tensor:
        return self.value_net(obs).squeeze(-1)


actor = GaussianActor(17, 6).to(DEVICE)
critic = ValueCritic(17).to(DEVICE)
batch_obs = torch.zeros(4, 17, device=DEVICE)
raw_action, old_log_prob, entropy = actor.sample(batch_obs)
value = critic(batch_obs)

assert raw_action.shape == (4, 6)
assert old_log_prob.shape == entropy.shape == value.shape == (4,)
print("raw action:", raw_action.shape, "joint log_prob:", old_log_prob.shape, "value:", value.shape)

### 动作边界的一致性

首版采用容易核查的方案：

1. Actor 采样未限制的 raw action `u`；
2. Buffer 保存 `u` 和 `log π_old(u|s)`；
3. 环境执行 `clip(u, -1, 1)`；
4. PPO 更新对同一个 `u` 计算 `log π_new(u|s)`。

它不是裁剪后有界动作的精确密度，因此后面专门记录动作饱和率。`tanh`-squashed Gaussian 与 Jacobian 修正可作为后续实验。

In [ ]:
with torch.no_grad():
    sampled_action, sampled_old_log_prob, _ = actor.sample(batch_obs)
same_policy_log_prob = actor.joint_log_prob(actor.distribution(batch_obs), sampled_action)
ratio_before_update = (same_policy_log_prob - sampled_old_log_prob).exp()
torch.testing.assert_close(ratio_before_update, torch.ones_like(ratio_before_update))
print("same policy / same raw action ratio:", ratio_before_update)

## 05｜TD、GAE 和 PPO-Clip

`terminated` 控制 TD bootstrap；`terminated or truncated` 控制 GAE 是否接到下一条样本。Rollout 最后一条即使 Episode 未结束，也停止数组内递推，但仍用自己的 `next_value`。

In [ ]:
@torch.no_grad()
def compute_gae(
    rewards: torch.Tensor,
    values: torch.Tensor,
    next_values: torch.Tensor,
    terminated: torch.Tensor,
    episode_ends: torch.Tensor,
    gamma: float,
    gae_lambda: float,
) -> tuple[torch.Tensor, torch.Tensor]:
    tensors = (rewards, values, next_values, terminated, episode_ends)
    if any(x.ndim != 1 for x in tensors):
        raise ValueError("all GAE inputs must be rank-1")
    if len({x.shape[0] for x in tensors}) != 1:
        raise ValueError("all GAE inputs must have equal length")

    bootstrap_mask = (~terminated.bool()).to(values.dtype)
    deltas = rewards + gamma * bootstrap_mask * next_values - values
    advantages = torch.zeros_like(rewards)
    gae = torch.zeros((), dtype=rewards.dtype, device=rewards.device)

    for t in range(rewards.shape[0] - 1, -1, -1):
        continue_mask = (~episode_ends[t].bool()).to(values.dtype)
        if t == rewards.shape[0] - 1:
            continue_mask = torch.zeros_like(continue_mask)
        gae = deltas[t] + gamma * gae_lambda * continue_mask * gae
        advantages[t] = gae

    returns = values + advantages
    return advantages.detach(), returns.detach()


def ppo_clipped_actor_loss(
    new_log_prob: torch.Tensor,
    old_log_prob: torch.Tensor,
    advantages: torch.Tensor,
    clip_coef: float,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    ratio = (new_log_prob - old_log_prob).exp()
    unclipped = ratio * advantages
    clipped = ratio.clamp(1 - clip_coef, 1 + clip_coef) * advantages
    loss = -torch.minimum(unclipped, clipped).mean()
    clip_fraction = ((ratio - 1).abs() > clip_coef).float().mean()
    return loss, ratio, clip_fraction


def value_loss(predicted: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    return 0.5 * (predicted - targets).pow(2).mean()

In [ ]:
# 手算测试 1：terminated 禁止 bootstrap
adv, ret = compute_gae(
    rewards=torch.tensor([1.0]), values=torch.tensor([0.25]),
    next_values=torch.tensor([10.0]), terminated=torch.tensor([True]),
    episode_ends=torch.tensor([True]), gamma=0.9, gae_lambda=0.95,
)
torch.testing.assert_close(adv, torch.tensor([0.75]))
torch.testing.assert_close(ret, torch.tensor([1.0]))

# 手算测试 2：truncated 允许 bootstrap，但切断递推
adv, ret = compute_gae(
    rewards=torch.tensor([1.0, 2.0]), values=torch.tensor([0.5, 0.25]),
    next_values=torch.tensor([4.0, 7.0]), terminated=torch.tensor([False, False]),
    episode_ends=torch.tensor([True, False]), gamma=0.9, gae_lambda=1.0,
)
torch.testing.assert_close(adv[0], torch.tensor(4.1))
torch.testing.assert_close(ret[0], torch.tensor(4.6))
assert not adv.requires_grad and not ret.requires_grad

# 手算测试 3：rollout 边界仍 bootstrap
adv_boundary, _ = compute_gae(
    rewards=torch.tensor([1.0]), values=torch.tensor([0.5]),
    next_values=torch.tensor([2.0]), terminated=torch.tensor([False]),
    episode_ends=torch.tensor([False]), gamma=0.9, gae_lambda=1.0,
)
torch.testing.assert_close(adv_boundary, torch.tensor([2.3]))

# Clip 对正/负 advantage 的方向测试
loss, ratio, clip_fraction = ppo_clipped_actor_loss(
    torch.log(torch.tensor([1.5, 1.5])), torch.zeros(2),
    torch.tensor([1.0, -1.0]), clip_coef=0.2,
)
torch.testing.assert_close(loss, torch.tensor(0.15))
torch.testing.assert_close(clip_fraction, torch.tensor(1.0))
print("GAE and PPO-Clip hand checks passed")

## 06｜Rollout Buffer 和环境交互

collector 持有当前观测和 Episode 累计量。达到 rollout 长度只切开训练 batch，**不会** reset 环境或清零 Episode Return/Length。`truncated` 的 `next_value` 在 reset 之前计算。

In [ ]:
@dataclass(slots=True)
class EpisodeRecord:
    episode_return: float
    episode_length: int
    terminated: bool
    truncated: bool
    forward_distance: float


@dataclass(slots=True)
class RolloutBatch:
    observations: torch.Tensor
    raw_actions: torch.Tensor
    old_log_probs: torch.Tensor
    old_values: torch.Tensor
    rewards: torch.Tensor
    next_values: torch.Tensor
    terminated: torch.Tensor
    episode_ends: torch.Tensor
    advantages: torch.Tensor
    returns: torch.Tensor
    reward_forward: torch.Tensor
    reward_ctrl: torch.Tensor
    reward_survive: torch.Tensor
    env_actions: torch.Tensor
    episodes: list[EpisodeRecord] = field(default_factory=list)

    def assert_valid(self, obs_dim: int, action_dim: int) -> None:
        n = self.rewards.shape[0]
        assert self.observations.shape == (n, obs_dim)
        assert self.raw_actions.shape == self.env_actions.shape == (n, action_dim)
        for name in (
            "old_log_probs", "old_values", "next_values", "terminated",
            "episode_ends", "advantages", "returns", "reward_forward",
            "reward_ctrl", "reward_survive",
        ):
            assert getattr(self, name).shape == (n,), name
        for tensor in (
            self.observations, self.raw_actions, self.old_log_probs, self.old_values,
            self.rewards, self.next_values, self.advantages, self.returns,
            self.reward_forward, self.reward_ctrl, self.reward_survive, self.env_actions,
        ):
            assert torch.isfinite(tensor).all()
            assert not tensor.requires_grad


class RolloutCollector:
    def __init__(
        self, env: gym.Env, actor: GaussianActor, critic: ValueCritic,
        device: torch.device, seed: int,
    ) -> None:
        self.env = env
        self.actor = actor
        self.critic = critic
        self.device = device
        obs, info = env.reset(seed=seed)
        self.obs = np.asarray(obs, dtype=np.float32)
        self.episode_return = 0.0
        self.episode_length = 0
        self.episode_start_x = float(info.get("x_position", 0.0))
        self.last_x = self.episode_start_x

    def collect(self, steps: int, gamma: float, gae_lambda: float) -> RolloutBatch:
        keys = (
            "observations", "raw_actions", "old_log_probs", "old_values",
            "rewards", "next_values", "terminated", "episode_ends",
            "reward_forward", "reward_ctrl", "reward_survive", "env_actions",
        )
        data: dict[str, list[Any]] = {key: [] for key in keys}
        episodes: list[EpisodeRecord] = []
        action_low = np.asarray(self.env.action_space.low, dtype=np.float32)
        action_high = np.asarray(self.env.action_space.high, dtype=np.float32)

        for _ in range(steps):
            obs_tensor = torch.as_tensor(
                self.obs, dtype=torch.float32, device=self.device
            ).unsqueeze(0)
            with torch.no_grad():
                raw_action_tensor, log_prob_tensor, _ = self.actor.sample(obs_tensor)
                value_tensor = self.critic(obs_tensor)

            raw_action = raw_action_tensor.squeeze(0).cpu().numpy()
            env_action = np.clip(raw_action, action_low, action_high).astype(np.float32)
            next_obs, reward, terminated, truncated, info = self.env.step(env_action)
            next_obs_f32 = np.asarray(next_obs, dtype=np.float32)

            # 必须在 reset 之前评价真正的 next_obs。
            with torch.no_grad():
                next_value = self.critic(torch.as_tensor(
                    next_obs_f32, dtype=torch.float32, device=self.device
                ).unsqueeze(0)).squeeze(0)

            data["observations"].append(obs_tensor.squeeze(0))
            data["raw_actions"].append(raw_action_tensor.squeeze(0))
            data["old_log_probs"].append(log_prob_tensor.squeeze(0))
            data["old_values"].append(value_tensor.squeeze(0))
            data["rewards"].append(float(reward))
            data["next_values"].append(next_value)
            data["terminated"].append(bool(terminated))
            data["episode_ends"].append(bool(terminated or truncated))
            data["reward_forward"].append(float(info["reward_forward"]))
            data["reward_ctrl"].append(float(info["reward_ctrl"]))
            data["reward_survive"].append(float(info["reward_survive"]))
            data["env_actions"].append(torch.as_tensor(env_action, device=self.device))

            self.episode_return += float(reward)
            self.episode_length += 1
            self.last_x = float(info.get("x_position", self.last_x))
            self.obs = next_obs_f32

            if terminated or truncated:
                episodes.append(EpisodeRecord(
                    episode_return=self.episode_return,
                    episode_length=self.episode_length,
                    terminated=bool(terminated), truncated=bool(truncated),
                    forward_distance=self.last_x - self.episode_start_x,
                ))
                reset_obs, reset_info = self.env.reset()
                self.obs = np.asarray(reset_obs, dtype=np.float32)
                self.episode_return = 0.0
                self.episode_length = 0
                self.episode_start_x = float(reset_info.get("x_position", 0.0))
                self.last_x = self.episode_start_x

        def stack(name: str, dtype: torch.dtype | None = None) -> torch.Tensor:
            items = data[name]
            if isinstance(items[0], torch.Tensor):
                result = torch.stack(items).detach()
            else:
                result = torch.as_tensor(items, device=self.device)
            return result.to(dtype=dtype) if dtype is not None else result

        rewards = stack("rewards", torch.float32)
        values = stack("old_values", torch.float32)
        next_values = stack("next_values", torch.float32)
        terminated_tensor = stack("terminated", torch.bool)
        episode_ends = stack("episode_ends", torch.bool)
        advantages, returns = compute_gae(
            rewards, values, next_values, terminated_tensor, episode_ends,
            gamma, gae_lambda,
        )

        return RolloutBatch(
            observations=stack("observations", torch.float32),
            raw_actions=stack("raw_actions", torch.float32),
            old_log_probs=stack("old_log_probs", torch.float32),
            old_values=values, rewards=rewards, next_values=next_values,
            terminated=terminated_tensor, episode_ends=episode_ends,
            advantages=advantages, returns=returns,
            reward_forward=stack("reward_forward", torch.float32),
            reward_ctrl=stack("reward_ctrl", torch.float32),
            reward_survive=stack("reward_survive", torch.float32),
            env_actions=stack("env_actions", torch.float32), episodes=episodes,
        )

In [ ]:
# 真实 Walker2d rollout 形状与有限值检查
env = gym.make("Walker2d-v5")
collector = RolloutCollector(env, actor, critic, DEVICE, seed=SEED)
rollout = collector.collect(steps=128, gamma=0.99, gae_lambda=0.95)
rollout.assert_valid(obs_dim=17, action_dim=6)
print({
    "observations": tuple(rollout.observations.shape),
    "raw_actions": tuple(rollout.raw_actions.shape),
    "joint_log_probs": tuple(rollout.old_log_probs.shape),
    "completed_episodes": len(rollout.episodes),
    "action_saturation": float((rollout.env_actions.abs() >= 0.999).float().mean()),
})
env.close()

In [ ]:
# 小型假环境：证明 rollout 边界不会错误 reset Episode
class LongEpisodeEnv(gym.Env):
    observation_space = spaces.Box(-np.inf, np.inf, shape=(2,), dtype=np.float32)
    action_space = spaces.Box(-1.0, 1.0, shape=(1,), dtype=np.float32)

    def __init__(self) -> None:
        self.t = 0
        self.reset_count = 0

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.t = 0
        self.reset_count += 1
        return np.zeros(2, dtype=np.float32), {"x_position": 0.0}

    def step(self, action):
        self.t += 1
        obs = np.array([self.t, 0], dtype=np.float32)
        info = {
            "x_position": float(self.t), "reward_forward": 0.0,
            "reward_ctrl": 0.0, "reward_survive": 1.0,
        }
        return obs, 1.0, self.t >= 10, False, info


fake_env = LongEpisodeEnv()
fake_actor, fake_critic = GaussianActor(2, 1), ValueCritic(2)
fake_collector = RolloutCollector(fake_env, fake_actor, fake_critic, DEVICE, seed=1)
first = fake_collector.collect(4, 0.99, 0.95)
second = fake_collector.collect(6, 0.99, 0.95)
assert not first.episodes
assert second.episodes[0].episode_length == 10
assert second.episodes[0].episode_return == 10.0
assert fake_env.reset_count == 2  # 初始 reset + 真正 Episode 结束
print("cross-rollout episode-state check passed")

## 07｜PPO 优化步骤

Rollout 保存的数据全部 detach。每个 mini-batch 只让当前 Actor 和当前 Critic 接收梯度，并记录 KL、Clip Fraction、Entropy 与动作饱和。

In [ ]:
@dataclass(slots=True)
class UpdateMetrics:
    actor_loss: float
    critic_loss: float
    approx_kl: float
    clip_fraction: float
    entropy: float
    action_mean: float
    action_std: float
    action_saturation: float


def ppo_update(
    actor: GaussianActor,
    critic: ValueCritic,
    actor_optimizer: torch.optim.Optimizer,
    critic_optimizer: torch.optim.Optimizer,
    batch: RolloutBatch,
    config: PPOConfig,
) -> UpdateMetrics:
    advantages = batch.advantages
    advantages = (advantages - advantages.mean()) / (advantages.std(unbiased=False) + 1e-8)
    n = advantages.shape[0]
    metrics: list[tuple[float, ...]] = []

    for _ in range(config.update_epochs):
        for indices in torch.randperm(n, device=advantages.device).split(config.minibatch_size):
            obs = batch.observations[indices]
            raw_actions = batch.raw_actions[indices]
            old_log_probs = batch.old_log_probs[indices]
            mb_advantages = advantages[indices]
            targets = batch.returns[indices]

            dist = actor.distribution(obs)
            new_log_probs = actor.joint_log_prob(dist, raw_actions)
            actor_loss, ratio, clip_fraction = ppo_clipped_actor_loss(
                new_log_probs, old_log_probs, mb_advantages, config.clip_coef
            )
            entropy = dist.entropy().sum(dim=-1).mean()
            total_actor_loss = actor_loss - config.entropy_coef * entropy
            actor_optimizer.zero_grad(set_to_none=True)
            total_actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(actor.parameters(), config.max_grad_norm)
            actor_optimizer.step()

            predicted_values = critic(obs)
            critic_loss = value_loss(predicted_values, targets)
            critic_optimizer.zero_grad(set_to_none=True)
            (config.value_coef * critic_loss).backward()
            torch.nn.utils.clip_grad_norm_(critic.parameters(), config.max_grad_norm)
            critic_optimizer.step()

            with torch.no_grad():
                log_ratio = new_log_probs - old_log_probs
                approx_kl = ((ratio - 1.0) - log_ratio).mean()
            metrics.append((
                float(actor_loss.detach()), float(critic_loss.detach()),
                float(approx_kl), float(clip_fraction), float(entropy.detach()),
            ))

    means = torch.tensor(metrics).mean(dim=0).tolist()
    with torch.no_grad():
        dist = actor.distribution(batch.observations)
        action_mean = float(dist.mean.mean())
        action_std = float(dist.stddev.mean())
        saturation = float((batch.env_actions.abs() >= 0.999).float().mean())
    return UpdateMetrics(*means, action_mean, action_std, saturation)

In [ ]:
# 用刚才的真实 rollout 做一次更新
smoke_config = profile("smoke")
actor_optimizer = torch.optim.Adam(actor.parameters(), lr=smoke_config.actor_lr)
critic_optimizer = torch.optim.Adam(critic.parameters(), lr=smoke_config.critic_lr)
update_metrics = ppo_update(
    actor, critic, actor_optimizer, critic_optimizer, rollout, smoke_config
)
print(update_metrics)
assert all(np.isfinite(v) for v in asdict(update_metrics).values())
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in actor.parameters())
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in critic.parameters())

## 08｜Checkpoint 与独立评估

评估使用 Actor 均值作为确定性动作，不采样、不更新参数。指标分别报告 Return、Length、前进距离和控制能量。

In [ ]:
@dataclass(slots=True)
class EvaluationResult:
    returns: list[float]
    lengths: list[int]
    distances: list[float]
    control_energy: list[float]

    def summary(self) -> dict[str, float]:
        result: dict[str, float] = {}
        for name, values in asdict(self).items():
            array = np.asarray(values, dtype=np.float64)
            result[f"mean_{name}"] = float(array.mean())
            result[f"std_{name}"] = float(array.std())
        return result


@torch.no_grad()
def evaluate_policy(
    actor: GaussianActor,
    env_id: str,
    seeds: list[int],
    device: torch.device,
    env_kwargs: dict[str, float] | None = None,
    video_path: Path | None = None,
) -> EvaluationResult:
    env = gym.make(
        env_id, render_mode="rgb_array" if video_path is not None else None,
        **(env_kwargs or {}),
    )
    returns, lengths, distances, energies, frames = [], [], [], [], []
    try:
        for episode_index, seed in enumerate(seeds):
            obs, info = env.reset(seed=seed)
            start_x = float(info.get("x_position", 0.0))
            last_x = start_x
            episode_return, episode_length, energy = 0.0, 0, 0.0
            while True:
                obs_tensor = torch.as_tensor(
                    obs, dtype=torch.float32, device=device
                ).unsqueeze(0)
                raw_action = actor.deterministic(obs_tensor).squeeze(0).cpu().numpy()
                action = np.clip(
                    raw_action, env.action_space.low, env.action_space.high
                ).astype(np.float32)
                obs, reward, terminated, truncated, info = env.step(action)
                episode_return += float(reward)
                episode_length += 1
                energy += float(np.square(action).sum())
                last_x = float(info.get("x_position", last_x))
                if video_path is not None and episode_index == 0:
                    frame = env.render()
                    if frame is not None:
                        frames.append(frame)
                if terminated or truncated:
                    break
            returns.append(episode_return)
            lengths.append(episode_length)
            distances.append(last_x - start_x)
            energies.append(energy)
    finally:
        env.close()

    if video_path is not None and frames:
        video_path.parent.mkdir(parents=True, exist_ok=True)
        imageio.mimsave(video_path, frames, fps=30, loop=0)
    return EvaluationResult(returns, lengths, distances, energies)


def save_checkpoint(
    path: Path,
    actor: GaussianActor,
    critic: ValueCritic,
    actor_optimizer: torch.optim.Optimizer,
    critic_optimizer: torch.optim.Optimizer,
    config: PPOConfig,
    env_steps: int,
    obs_dim: int,
    action_dim: int,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "actor": actor.state_dict(), "critic": critic.state_dict(),
        "actor_optimizer": actor_optimizer.state_dict(),
        "critic_optimizer": critic_optimizer.state_dict(),
        "config": config.as_serializable_dict(), "env_steps": env_steps,
        "seed": config.seed, "obs_dim": obs_dim, "action_dim": action_dim,
        "torch_rng_state": torch.get_rng_state(),
        "numpy_rng_state": np.random.get_state(),
        "note": "Exact mid-episode resume is not implemented.",
    }, path)


def load_actor(path: Path, device: torch.device) -> tuple[GaussianActor, dict[str, Any]]:
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    config = checkpoint["config"]
    actor = GaussianActor(
        int(checkpoint["obs_dim"]), int(checkpoint["action_dim"]),
        int(config["hidden_size"]),
    ).to(device)
    actor.load_state_dict(checkpoint["actor"])
    actor.eval()
    return actor, checkpoint

In [ ]:
# checkpoint 往返一致性测试
test_path = Path("checkpoints/roundtrip_test.pt")
test_path.parent.mkdir(parents=True, exist_ok=True)
test_obs = torch.randn(3, 17)
before = actor.deterministic(test_obs).detach()
save_checkpoint(
    test_path, actor, critic, actor_optimizer, critic_optimizer,
    smoke_config, env_steps=128, obs_dim=17, action_dim=6,
)
restored_actor, restored_checkpoint = load_actor(test_path, DEVICE)
after = restored_actor.deterministic(test_obs).detach()
torch.testing.assert_close(before, after)
assert restored_checkpoint["env_steps"] == 128
print("checkpoint roundtrip check passed")

## 09｜完整训练函数和 CSV 日志

每个 rollout/update 写一行，横轴 `env_steps` 是真实环境交互数。同一 rollout 内多个 epoch/mini-batch 的指标聚合到这一行。

In [ ]:
METRIC_FIELDS = [
    "env_steps", "update", "steps_per_second", "update_seconds",
    "actor_loss", "critic_loss", "approx_kl", "clip_fraction", "entropy",
    "action_mean", "action_std", "action_saturation", "mean_rollout_reward",
    "mean_reward_forward", "mean_reward_ctrl", "mean_reward_survive",
    "completed_episodes", "mean_episode_return", "mean_episode_length",
    "termination_fraction", "truncation_fraction",
]


def safe_mean(values: list[float]) -> float:
    return float(np.mean(values)) if values else float("nan")


def train(config: PPOConfig) -> Path:
    config.validate()
    seed_everything(config.seed)
    device = torch.device(config.device)
    output_dir = Path(config.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    (output_dir / "config.json").write_text(
        json.dumps(config.as_serializable_dict(), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )

    env = gym.make(config.env_id, **config.env_kwargs())
    obs_dim = int(np.prod(env.observation_space.shape))
    action_dim = int(np.prod(env.action_space.shape))
    actor = GaussianActor(obs_dim, action_dim, config.hidden_size).to(device)
    critic = ValueCritic(obs_dim, config.hidden_size).to(device)
    actor_optimizer = torch.optim.Adam(actor.parameters(), lr=config.actor_lr)
    critic_optimizer = torch.optim.Adam(critic.parameters(), lr=config.critic_lr)
    collector = RolloutCollector(env, actor, critic, device, config.seed)

    metrics_path = output_dir / "metrics.csv"
    env_steps, update_index = 0, 0
    start_time = time.perf_counter()
    try:
        with metrics_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=METRIC_FIELDS)
            writer.writeheader()
            while env_steps < config.total_timesteps:
                steps = min(config.rollout_steps, config.total_timesteps - env_steps)
                batch = collector.collect(steps, config.gamma, config.gae_lambda)
                batch.assert_valid(obs_dim, action_dim)
                env_steps += steps

                update_started = time.perf_counter()
                update_metrics = ppo_update(
                    actor, critic, actor_optimizer, critic_optimizer, batch, config
                )
                update_seconds = time.perf_counter() - update_started
                update_index += 1
                elapsed = time.perf_counter() - start_time

                episode_returns = [ep.episode_return for ep in batch.episodes]
                episode_lengths = [float(ep.episode_length) for ep in batch.episodes]
                completed = len(batch.episodes)
                row = {
                    "env_steps": env_steps, "update": update_index,
                    "steps_per_second": env_steps / max(elapsed, 1e-9),
                    "update_seconds": update_seconds, **asdict(update_metrics),
                    "mean_rollout_reward": float(batch.rewards.mean()),
                    "mean_reward_forward": float(batch.reward_forward.mean()),
                    "mean_reward_ctrl": float(batch.reward_ctrl.mean()),
                    "mean_reward_survive": float(batch.reward_survive.mean()),
                    "completed_episodes": completed,
                    "mean_episode_return": safe_mean(episode_returns),
                    "mean_episode_length": safe_mean(episode_lengths),
                    "termination_fraction": (
                        sum(ep.terminated for ep in batch.episodes) / completed
                        if completed else float("nan")
                    ),
                    "truncation_fraction": (
                        sum(ep.truncated for ep in batch.episodes) / completed
                        if completed else float("nan")
                    ),
                }
                writer.writerow(row)
                handle.flush()
    finally:
        env.close()

    checkpoint_path = output_dir / "final.pt"
    save_checkpoint(
        checkpoint_path, actor, critic, actor_optimizer, critic_optimizer,
        config, env_steps, obs_dim, action_dim,
    )
    evaluation = evaluate_policy(
        actor, config.env_id,
        seeds=[10_000 + config.seed + i for i in range(config.eval_episodes)],
        device=device, env_kwargs=config.env_kwargs(),
    )
    (output_dir / "evaluation.json").write_text(
        json.dumps(evaluation.summary(), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    return checkpoint_path

## 10｜运行 smoke、查看仪表盘

默认执行 128 个环境步和一次 PPO update。想观察训练趋势时，把 `RUN_PROFILE` 改成 `short`；运行 `full` 前先确认本机吞吐。

In [ ]:
RUN_PROFILE = "smoke"  # 可改为 short / full
run_dir = Path(f"runs/notebook-{RUN_PROFILE}")
checkpoint_path = train(profile(RUN_PROFILE, output_dir=run_dir))
print("checkpoint:", checkpoint_path)

metrics_data = np.atleast_1d(np.genfromtxt(
    run_dir / "metrics.csv", delimiter=",", names=True
))
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(metrics_data["env_steps"], metrics_data["mean_rollout_reward"])
axes[0].set_title("Mean step reward")
axes[1].plot(metrics_data["env_steps"], metrics_data["actor_loss"], label="actor")
axes[1].plot(metrics_data["env_steps"], metrics_data["critic_loss"], label="critic")
axes[1].legend(); axes[1].set_title("Update losses")
axes[2].plot(metrics_data["env_steps"], metrics_data["action_saturation"])
axes[2].set_title("Executed action saturation")
for ax in axes:
    ax.set_xlabel("environment steps")
plt.tight_layout(); plt.show()

## 11｜固定 seed 独立评估与回放

下面不会执行梯度更新。smoke 策略通常很快摔倒，这是正确且诚实的预期。

In [ ]:
trained_actor, saved = load_actor(checkpoint_path, DEVICE)
evaluation_gif = Path(f"outputs/{RUN_PROFILE}-evaluation.gif")
evaluation = evaluate_policy(
    trained_actor, "Walker2d-v5", seeds=[10000, 10001, 10002],
    device=DEVICE,
    env_kwargs={
        "forward_reward_weight": saved["config"]["forward_reward_weight"],
        "ctrl_cost_weight": saved["config"]["ctrl_cost_weight"],
        "healthy_reward": saved["config"]["healthy_reward"],
    },
    video_path=evaluation_gif,
)
print(json.dumps(evaluation.summary(), indent=2, ensure_ascii=False))
print("video:", evaluation_gif)

## 12｜奖励权重对照实验

先写假设，再一次只改一个权重。不同奖励定义下的训练 Return 不能直接横比；统一比较 Episode Length、前进距离、控制能量、动作饱和与视频。

推荐 C1：`ctrl_cost_weight ∈ {1e-4, 1e-3, 1e-2}`。控制惩罚增大可能降低动作能量，也可能削弱恢复平衡的能力。

In [ ]:
experiment_configs = {
    "low_ctrl_cost": profile(
        "short", ctrl_cost_weight=1e-4, output_dir=Path("runs/c1/low")
    ),
    "baseline": profile(
        "short", ctrl_cost_weight=1e-3, output_dir=Path("runs/c1/base")
    ),
    "high_ctrl_cost": profile(
        "short", ctrl_cost_weight=1e-2, output_dir=Path("runs/c1/high")
    ),
}
for name, cfg in experiment_configs.items():
    print(name, cfg.ctrl_cost_weight, cfg.total_timesteps, cfg.seed)

# 明确取消下一行注释后才会依次训练：
# experiment_checkpoints = {name: train(cfg) for name, cfg in experiment_configs.items()}

## 13｜纯仿真鲁棒性扩展

基线稳定后，再固定策略并改变测试时的地面摩擦或连杆质量，比较标称、0.9×、1.0×、1.1× 条件下的距离、Length 和能量。先记录训练/测试分布，不要直接加入复杂域随机化。

这些结果只说明仿真内泛化，不等于真实 Sim-to-Real。真实硬件还包含传感器噪声、执行器带宽、延迟、柔性和未建模接触。

### 最终自检

- 能否解释 17 维观测与 6 维力矩？
- 能否画出 raw action → clip → environment 与 log probability 的数据流？
- 能否说明 `terminated` 与 `truncated` 对应的两个 mask？
- 能否从日志和视频区分代码错误、训练不足与奖励设计问题？
- 能否提出一次只改变一个因素、使用共同外部指标的实验？